# IOAI — 2025 Stage 3 Unlearning (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
if not os.path.exists('data/lenet_base_final.pt'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2025-stage-3-unlearning/data.zip', 'd.zip')
    zipfile.ZipFile('d.zip').extractall('data')
print('데이터:', sorted(os.listdir('data')))
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# 기계 오둔학습 (Machine Unlearning, 베이스라인)

폴란드 AI 올림피아드 II · 2025 · 결선. Fashion-MNIST 전체(10클래스)로 학습된 **LeNet** 에서, 특정 클래스
(여기선 **9번=앵클부츠**)를 **잊게(unlearn)** 만든다. 단, **마지막 분류층 `fc2` 는 건드리지 않고**(특징
추출부만 수정), 나머지 클래스 성능은 유지해야 한다.

**채점**(4항목 각 25점): ① 잊을 클래스 예측이 균등분포에 가까움(KL↓ [0.2→0.5]) ② 나머지 정확도↑ [0.87→0.90]
③ 원본과의 L2 변화↓ [1.3→3.0] ④ 잊을 클래스 정확도↓ [0.09→0.3]. 하드제약(rest≥0.75, target≤0.5, L2≤8, KL≤1.75)
위반 시 0.

이 노트북은 **베이스라인** = `unlearn` 이 모델을 그대로 반환(오둔학습 안 함) → 잊을 클래스 정확도 0.96 → **0점**.
모범답안(특징추출부 미세조정)을 참고하라.

**제출**: `submission.pt` — 오둔학습된 모델 state_dict.


In [ ]:
# 데이터 준비 (Colab: 자동 다운로드 / DGX: data/ 이미 존재)
import os, copy, urllib.request, zipfile
if not os.path.exists("data/lenet_base_final.pt"):
    url = "https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2025-stage-3-unlearning/data.zip"
    urllib.request.urlretrieve(url, "d.zip"); zipfile.ZipFile("d.zip").extractall("data")

import torch, torch.nn as nn, torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"; TARGET_CLASS = 9

class LeNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.block1 = nn.Sequential(nn.Conv2d(1,6,5,1,0), nn.BatchNorm2d(6), nn.ReLU(), nn.MaxPool2d(2,2))
        self.block2 = nn.Sequential(nn.Conv2d(6,16,5,1,0), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2,2))
        self.fc = nn.Linear(256,120); self.relu = nn.ReLU(); self.fc1 = nn.Linear(120,84); self.relu1 = nn.ReLU(); self.fc2 = nn.Linear(84,10)
    def forward(self, x):
        o=self.block1(x); o=self.block2(o); o=o.reshape(o.size(0),-1)
        return self.fc2(self.relu1(self.fc1(self.relu(self.fc(o)))))

pretrained_model = LeNet().to(DEVICE)
pretrained_model.load_state_dict(torch.load("data/lenet_base_final.pt", map_location=DEVICE)); pretrained_model.eval()

tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.2860,),(0.3530,))])
test = datasets.FashionMNIST("data", train=False, download=False, transform=tf)
tgt_loader  = DataLoader(Subset(test, (test.targets==TARGET_CLASS).nonzero().flatten().tolist()), batch_size=256)
rest_loader = DataLoader(Subset(test, (test.targets!=TARGET_CLASS).nonzero().flatten().tolist()), batch_size=256)
print("target(9)", len(tgt_loader.dataset), "rest", len(rest_loader.dataset), "| device", DEVICE)


In [ ]:
def unlearn(model, target_class=TARGET_CLASS):
    """베이스라인: 오둔학습 없이 모델을 그대로 반환 (잊을 클래스 정확도 유지 → 0점). TODO 구현."""
    return copy.deepcopy(model)

unlearned_model = unlearn(pretrained_model)


In [ ]:
# fc2 불변 확인 + submission.pt 저장
assert all(torch.equal(a.cpu(), b.cpu()) for a, b in zip(unlearned_model.fc2.parameters(), pretrained_model.fc2.parameters())), "fc2 가 변경됨!"
torch.save(unlearned_model.state_dict(), "submission.pt")
print("submission.pt 저장 완료")


### 다음 단계
그대로 두면 잊을 클래스 정확도가 0.96 → 0점. `unlearn` 을 **특징추출부 미세조정**(fc2 동결, 잊을 클래스는
균등분포로, 나머지는 유지, L2 억제)으로 구현하면 잊을 acc 0.09·나머지 0.89 → ~90점. 모범답안 참고.


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.pt']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)